In [4]:
from pyspark.sql import SparkSession

# Create Spark session
spark = SparkSession.builder \
    .appName("TestGlobalView") \
    .getOrCreate()

```text
RDD (Resilient Distributed Dataset) is Spark's low-level distributed data structure.


RDD  -->  DataFrame  -->  Spark SQL
(low level)  (structured)  (SQL layer)





When to use RDD?

RDDs are now used much less frequently because DataFrames are:

Faster (Catalyst Optimizer + Tungsten)
Easier to write
Support SQL
Better memory optimization

Most modern Spark code uses DataFrames and Spark SQL. Use RDDs mainly when:

You need custom low-level transformations.
You're working with unstructured data.
You're learning Spark internals.

For interviews, a common answer is:

RDD is the fundamental distributed collection in Spark. DataFrames are built on top of RDDs and are generally preferred because they provide schema information and Spark can optimize their execution.



```

In [ ]:
# Convert RDD to DataFrame
data    = [('surya','john'),('marco','paul')]

rdd     = spark.sparkContext.parallelize(data)
df      = rdd.toDF(['name','surname'])   # it is DataFrame
df
df.show()
display(df)

+-----+-------+
| name|surname|
+-----+-------+
|surya|   john|
|marco|   paul|
+-----+-------+



DataFrame[name: string, surname: string]

```text

parallelize() is a method of SparkContext that converts a local Python collection (list, tuple, etc.) into an RDD.

Why "parallelize"?

Spark splits the data into partitions so it can process them in parallel across cores (or across a cluster).




Common Interview Question

Q: What is parallelize()?

A: parallelize() is a SparkContext method that distributes a local collection into an RDD, allowing Spark to process the data in parallel across partitions.



```

In [16]:
data = [
    (1, 'Michael', 10000, 1000),
    (2, 'Andy', 20000, 2000),
    (3, 'Justin', 30000, 3000),
    (4, 'Berta', 40000, 4000)
]

rdd = spark.sparkContext.parallelize(data)



print(type(rdd))
print(rdd.collect())
df = rdd.toDF(['id','name','salary','bonus'])
df.show()

<class 'pyspark.core.rdd.RDD'>
[(1, 'Michael', 10000, 1000), (2, 'Andy', 20000, 2000), (3, 'Justin', 30000, 3000), (4, 'Berta', 40000, 4000)]
+---+-------+------+-----+
| id|   name|salary|bonus|
+---+-------+------+-----+
|  1|Michael| 10000| 1000|
|  2|   Andy| 20000| 2000|
|  3| Justin| 30000| 3000|
|  4|  Berta| 40000| 4000|
+---+-------+------+-----+



In [21]:
# way 1

from pyspark.sql.functions import col, concat_ws

data = [('surya','john'),('marco','paul')]
df = spark.createDataFrame(data, ['fn','ln'])

# Best Practice: Use native DataFrame operations
df1 = df.withColumn('full_name', concat_ws(' ', col('fn'), col('ln')))

df1.show()



+-----+----+----------+
|   fn|  ln| full_name|
+-----+----+----------+
|surya|john|surya john|
|marco|paul|marco paul|
+-----+----+----------+



In [22]:
data = [('surya','john'),('marco','paul')]
df = spark.createDataFrame(data, ['fn','ln'])

# FIX: Explicitly create a brand new standard tuple inside the lambda
rdd1 = df.rdd.map(
    lambda x: (x.fn, x.ln, x.fn + ' ' + x.ln) 
)

# Now Spark can easily read this clean tuple into 3 columns
df1 = spark.createDataFrame(rdd1, ['fn', 'ln', 'full_name'])
df1.show()

Py4JJavaError: An error occurred while calling z:org.apache.spark.api.python.PythonRDD.runJob.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 11 in stage 84.0 failed 1 times, most recent failure: Lost task 11.0 in stage 84.0 (TID 676) (MasterTrader executor driver): org.apache.spark.SparkException: Python worker exited unexpectedly (crashed). Consider setting 'spark.sql.execution.pyspark.udf.faulthandler.enabled' or'spark.python.worker.faulthandler.enabled' configuration to 'true' for the better Python traceback.
	at org.apache.spark.api.python.BasePythonRunner$ReaderIterator$$anonfun$1.applyOrElse(PythonRunner.scala:678)
	at org.apache.spark.api.python.BasePythonRunner$ReaderIterator$$anonfun$1.applyOrElse(PythonRunner.scala:663)
	at scala.runtime.AbstractPartialFunction.apply(AbstractPartialFunction.scala:35)
	at org.apache.spark.api.python.PythonRunner$$anon$3.read(PythonRunner.scala:1034)
	at org.apache.spark.api.python.PythonRunner$$anon$3.read(Pyth

In [ ]:
# wway 2 Rdd ways


data = [('surya','john'),('marco','paul')]
df = spark.createDataFrame(data, ['fn','ln'])

rdd1 = df.rdd.map(
    lambda x: x + (x[0] + ' ' + x[1],)
)

# FIXED: Provided exactly 3 column names that match the data
df1 = spark.createDataFrame(rdd1, ['fn', 'ln', 'full_name']) 
df1.show()

Py4JJavaError: An error occurred while calling z:org.apache.spark.api.python.PythonRDD.runJob.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 10 in stage 78.0 failed 1 times, most recent failure: Lost task 10.0 in stage 78.0 (TID 631) (MasterTrader executor driver): org.apache.spark.SparkException: Python worker exited unexpectedly (crashed). Consider setting 'spark.sql.execution.pyspark.udf.faulthandler.enabled' or'spark.python.worker.faulthandler.enabled' configuration to 'true' for the better Python traceback.
	at org.apache.spark.api.python.BasePythonRunner$ReaderIterator$$anonfun$1.applyOrElse(PythonRunner.scala:678)
	at org.apache.spark.api.python.BasePythonRunner$ReaderIterator$$anonfun$1.applyOrElse(PythonRunner.scala:663)
	at scala.runtime.AbstractPartialFunction.apply(AbstractPartialFunction.scala:35)
	at org.apache.spark.api.python.PythonRunner$$anon$3.read(PythonRunner.scala:1034)
	at org.apache.spark.api.python.PythonRunner$$anon$3.read(Pyth

In [11]:
data = [('surya','john'),('marco','paul')]

df = spark.createDataFrame(data, ['fn','ln'])

rdd1 = df.rdd.map(
    lambda x: x + (x[0] + ' ' + x[1],)
)

df1 = rdd1.toDF(schema=['fn', 'ln', 'fullname'])

df1.show()

Py4JJavaError: An error occurred while calling z:org.apache.spark.api.python.PythonRDD.runJob.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 8 in stage 35.0 failed 1 times, most recent failure: Lost task 8.0 in stage 35.0 (TID 255) (MasterTrader executor driver): org.apache.spark.SparkException: Python worker exited unexpectedly (crashed). Consider setting 'spark.sql.execution.pyspark.udf.faulthandler.enabled' or'spark.python.worker.faulthandler.enabled' configuration to 'true' for the better Python traceback.
	at org.apache.spark.api.python.BasePythonRunner$ReaderIterator$$anonfun$1.applyOrElse(PythonRunner.scala:678)
	at org.apache.spark.api.python.BasePythonRunner$ReaderIterator$$anonfun$1.applyOrElse(PythonRunner.scala:663)
	at scala.runtime.AbstractPartialFunction.apply(AbstractPartialFunction.scala:35)
	at org.apache.spark.api.python.PythonRunner$$anon$3.read(PythonRunner.scala:1034)
	at org.apache.spark.api.python.PythonRunner$$anon$3.read(Python

In [8]:
data = [('surya','john'),('marco','paul')]

df = spark.createDataFrame(data, ['fn','ln'])
df.show()

rdd1 = df.rdd.map(lambda x: x + (x[0] + ' ' + x[1],))

# df1 = rdd1.toDF(*['fn', 'ln', 'fullname'])
df1 = rdd1.toDF('fn', 'ln', 'fullname')
df1.show()

+-----+----+
|   fn|  ln|
+-----+----+
|surya|john|
|marco|paul|
+-----+----+



TypeError: _monkey_patch_RDD.<locals>.toDF() takes from 1 to 3 positional arguments but 4 were given

In [0]:
data = [('surya','john'),('marco','paul' )]

def add_fullname(x):
    return x + (x[0] + ' '+x[1],)

# rdd are immutable, so we need to create a new rdd with the data
df      =       spark.createDataFrame(data, ['fn','ln'])
df.show()

rdd1    =       df.rdd.map(lambda x : add_fullname(x))
df1     =       rdd1.toDF(['fn','ln','fullname'])
df1.show()


# flatmap

RDD (Resilient Distributed Dataset) is Spark's fundamental distributed data structure. It is an immutable, fault-tolerant collection of data partitioned across a cluster and processed in parallel.

In [0]:
data = [('surya john'),('marco paul' )]

# rdd are immutable, so we need to create a new rdd with the data
rdd = spark.sparkContext.parallelize(data)


rdd1 = rdd.flatMap(lambda x: x.split(' '))
print(rdd1.collect())

for item in rdd1.collect():
    print(item)

    

### Output

```text
['surya', 'john', 'marco', 'paul']

surya
john
marco
paul
```
